In [1]:
import requests
import csv
import time
import json
from pathlib import Path

# === Configuration ===
BASE_URL = "https://www.loc.gov/collections/chronicling-america/"
API_BASE = "https://www.loc.gov"

TARGET_YEARS = [1920, 1921, 1922]

# Output paths
OUTPUT_DIR = Path(r"D:\Shikha\bias_project\corpus\chronicling_america\1920-1922")
ARTICLES_CSV = OUTPUT_DIR / "articles.csv"
PROGRESS_LOG = OUTPUT_DIR / "progress.log"
ERRORS_LOG  = OUTPUT_DIR / "errors.log"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Rate limiting — be polite to loc.gov
REQUEST_DELAY = 1.0
MAX_RETRIES   = 3
RETRY_WAIT    = 15

print(f"Output directory : {OUTPUT_DIR}")
print(f"Output exists    : {OUTPUT_DIR.exists()}")
print(f"Target years     : {TARGET_YEARS}")

Output directory : D:\Shikha\bias_project\corpus\chronicling_america\1920-1922
Output exists    : True
Target years     : [1920, 1921, 1922]


In [ ]:
# Check which states have Chronicling America coverage for 1920-1922

US_STATES = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR',
    'california': 'CA', 'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE',
    'florida': 'FL', 'georgia': 'GA', 'hawaii': 'HI', 'idaho': 'ID',
    'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA', 'kansas': 'KS',
    'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN', 'mississippi': 'MS',
    'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV',
    'new hampshire': 'NH', 'new jersey': 'NJ', 'new mexico': 'NM', 'new york': 'NY',
    'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK',
    'oregon': 'OR', 'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC',
    'south dakota': 'SD', 'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT',
    'vermont': 'VT', 'virginia': 'VA', 'washington': 'WA', 'west virginia': 'WV',
    'wisconsin': 'WI', 'wyoming': 'WY'
}

def check_state_coverage(state_slug, year):
    url = f"{API_BASE}/collections/chronicling-america/"
    params = {
        'fa': f'location_state:{state_slug}',
        'dates': f'{year}/{year}',
        'fo': 'json',
        'c': 1  # just need count, not full results
    }
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200:
                data = r.json()
                count = data.get('pagination', {}).get('total', 0)
                return count
            elif r.status_code == 429:
                time.sleep(RETRY_WAIT)
        except Exception as e:
            time.sleep(RETRY_WAIT)
    return -1  # error

# Sample a representative set of states first
SAMPLE_STATES = {
    'new-york': 'NY', 'ohio': 'OH', 'wyoming': 'WY',
    'california': 'CA', 'texas': 'TX', 'illinois': 'IL',
    'mississippi': 'MS', 'montana': 'MT', 'oregon': 'OR'
}

print(f"{'State':<20} {'1920':>8} {'1921':>8} {'1922':>8} {'Total':>8}")
print("-" * 50)

coverage = {}
for state_slug, abbr in SAMPLE_STATES.items():
    counts = []
    for yr in TARGET_YEARS:
        n = check_state_coverage(state_slug, yr)
        counts.append(n)
        time.sleep(REQUEST_DELAY)
    total = sum(c for c in counts if c > 0)
    coverage[abbr] = counts
    print(f"{abbr:<20} {counts[0]:>8} {counts[1]:>8} {counts[2]:>8} {total:>8}")

print("\nDone. States with 0 pages will be skipped in extraction.")

State                    1920     1921     1922    Total
--------------------------------------------------
NY                          1        1        1        3
OH                       1569     1593     1239     4401
WY                       1464     1432     1675     4571
CA                        429      427      435     1291
TX                       1693     1356     1118     4167
IL                       2312     2140     2139     6591
MS                       1076     1045      929     3050
MT                       1001      712      823     2536
OR                       1582     1448     1463     4493

Done. States with 0 pages will be skipped in extraction.


In [3]:
# Test different slug formats for New York
ny_variants = [
    'new-york',
    'new_york', 
    'newyork',
    'New York',
    'new york',
]

for slug in ny_variants:
    url = f"{API_BASE}/collections/chronicling-america/"
    params = {
        'fa': f'location_state:{slug}',
        'dates': '1920/1920',
        'fo': 'json',
        'c': 1
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code == 200:
            data = r.json()
            count = data.get('pagination', {}).get('total', 0)
            print(f"'{slug}' → {count} pages")
        else:
            print(f"'{slug}' → HTTP {r.status_code}")
    except Exception as e:
        print(f"'{slug}' → ERROR: {e}")
    time.sleep(1)

'new-york' → 1 pages
'new_york' → ERROR: HTTPSConnectionPool(host='www.loc.gov', port=443): Read timed out.
'newyork' → HTTP 503
'New York' → 1427 pages
'new york' → 1427 pages


In [10]:
# Fetch page-level OCR text — text is in 'description' field directly

CONFIRMED_STATES = {
    'ohio': 'OH', 'wyoming': 'WY', 'california': 'CA',
    'texas': 'TX', 'illinois': 'IL', 'mississippi': 'MS',
    'montana': 'MT', 'oregon': 'OR', 'new york': 'NY'
}

PAGES_PER_CELL = 50

def fetch_pages_for_state_year(state_slug, year, n=50):
    url = "https://www.loc.gov/collections/chronicling-america/"
    params = {
        'fa':         f'location_state:{state_slug}',
        'start_date': f'{year}-01-01',
        'end_date':   f'{year}-12-31',
        'dl':         'page',
        'fo':         'json',
        'c':          min(n, 25),
        'at':         'results,pagination'
    }
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:
                time.sleep(RETRY_WAIT)
        except:
            time.sleep(5)
    return None

def get_ocr_from_item(item):
    # OCR text is in 'description' field as a list — join into one string
    desc = item.get('description', [])
    if isinstance(desc, list):
        text = ' '.join(str(d) for d in desc)
    else:
        text = str(desc)
    return text.strip() if len(text) > 100 else None

# Open output CSV
fieldnames = ['state', 'state_abbr', 'year', 'item_id',
              'title', 'date', 'text_length', 'text']
f = open(ARTICLES_CSV, 'w', newline='', encoding='utf-8')
writer = csv.DictWriter(f, fieldnames=fieldnames)
writer.writeheader()
err_f = open(ERRORS_LOG, 'w', encoding='utf-8')

stats = {'fetched': 0, 'no_text': 0, 'errors': 0}

for state_slug, abbr in CONFIRMED_STATES.items():
    for year in TARGET_YEARS:
        print(f"\n[{abbr} {year}] fetching pages...")
        fetched_this_cell = 0
        api_page = 1

        while fetched_this_cell < PAGES_PER_CELL:
            data = fetch_pages_for_state_year(state_slug, year, 25)
            if not data:
                print(f"  → API error, skipping")
                break

            results = data.get('results', [])
            if not results:
                break

            for item in results:
                if fetched_this_cell >= PAGES_PER_CELL:
                    break
                text = get_ocr_from_item(item)
                if text:
                    writer.writerow({
                        'state':      state_slug,
                        'state_abbr': abbr,
                        'year':       year,
                        'item_id':    item.get('aka', [''])[0],
                        'title':      item.get('title', ''),
                        'date':       item.get('date', ''),
                        'text_length': len(text),
                        'text':       text
                    })
                    fetched_this_cell += 1
                    stats['fetched'] += 1
                else:
                    stats['no_text'] += 1
                time.sleep(REQUEST_DELAY)

            api_page += 1
            # avoid infinite loop if API keeps returning same page
            if len(results) < 25:
                break

        print(f"  → {fetched_this_cell} pages saved for {abbr} {year}")

f.close()
err_f.close()
print(f"\n--- Done ---")
print(f"Total fetched : {stats['fetched']}")
print(f"No text       : {stats['no_text']}")
print(f"Output        : {ARTICLES_CSV}")


[OH 1920] fetching pages...
  → 50 pages saved for OH 1920

[OH 1921] fetching pages...
  → 50 pages saved for OH 1921

[OH 1922] fetching pages...
  → 50 pages saved for OH 1922

[WY 1920] fetching pages...
  → 50 pages saved for WY 1920

[WY 1921] fetching pages...
  → 50 pages saved for WY 1921

[WY 1922] fetching pages...
  → 50 pages saved for WY 1922

[CA 1920] fetching pages...
  → 50 pages saved for CA 1920

[CA 1921] fetching pages...
  → 50 pages saved for CA 1921

[CA 1922] fetching pages...
  → 50 pages saved for CA 1922

[TX 1920] fetching pages...
  → 50 pages saved for TX 1920

[TX 1921] fetching pages...
  → 50 pages saved for TX 1921

[TX 1922] fetching pages...
  → 50 pages saved for TX 1922

[IL 1920] fetching pages...
  → 50 pages saved for IL 1920

[IL 1921] fetching pages...
  → 50 pages saved for IL 1921

[IL 1922] fetching pages...
  → 50 pages saved for IL 1922

[MS 1920] fetching pages...
  → 50 pages saved for MS 1920

[MS 1921] fetching pages...
  → 50 page

In [5]:
# Inspect raw API response for OH 1920 — see actual item URL structure
url = f"{API_BASE}/collections/chronicling-america/"
params = {
    'fa': 'location_state:ohio',
    'dates': '1920/1920',
    'fo': 'json',
    'c': 3
}

r = requests.get(url, params=params, timeout=15)
data = r.json()

results = data.get('results', [])
print(f"Total results in API: {data.get('pagination', {}).get('total', 0)}")
print(f"Results returned: {len(results)}")
print()

for i, item in enumerate(results):
    print(f"--- Item {i+1} ---")
    print(f"  url    : {item.get('url', 'MISSING')}")
    print(f"  id     : {item.get('id', 'MISSING')}")
    print(f"  title  : {item.get('title', 'MISSING')}")
    print(f"  date   : {item.get('date', 'MISSING')}")
    print(f"  keys   : {list(item.keys())}")
    print()

Total results in API: 523
Results returned: 3

--- Item 1 ---
  url    : https://www.loc.gov/item/sn89075140/
  id     : http://www.loc.gov/item/sn89075140/
  title  : The Tri-County News (Grand Rapids, Ohio) 1915-1949
  date   : 1915
  keys   : ['access_restricted', 'aka', 'batch', 'campaigns', 'composite_location', 'date', 'description', 'digitized', 'extract_timestamp', 'group', 'hassegments', 'id', 'image_url', 'index', 'item', 'language', 'latlong', 'location', 'location_city', 'location_country', 'location_county', 'location_state', 'location_str', 'lonlat', 'number', 'number_first_issue', 'number_hasessay', 'number_holding_type', 'number_institution', 'number_issn', 'number_issue_count', 'number_last_issue', 'number_lccn', 'number_oclc', 'number_preceding_items', 'number_source_modified', 'original_format', 'partof', 'partof_collection', 'partof_division', 'partof_title', 'resources', 'shelf_id', 'site', 'subject', 'timestamp', 'title', 'type', 'url']

--- Item 2 ---
  url    : 

In [6]:
# Use the LCCN to get actual dated issues and their OCR text
# Test with first result: sn89075140 (Tri-County News, Ohio)

LCCN = 'sn89075140'
TARGET_YEAR = 1920

# Step 1: get issues for this title in 1920
issues_url = f"https://chroniclingamerica.loc.gov/lccn/{LCCN}/issues/{TARGET_YEAR}.json"
r = requests.get(issues_url, timeout=15)
print(f"Issues URL: {issues_url}")
print(f"Status: {r.status_code}")

if r.status_code == 200:
    data = r.json()
    issues = data.get('issues', [])
    print(f"Issues found in {TARGET_YEAR}: {len(issues)}")
    if issues:
        print(f"First issue: {issues[0]}")
        print(f"Last issue:  {issues[-1]}")
else:
    print(f"Response: {r.text[:300]}")

Issues URL: https://chroniclingamerica.loc.gov/lccn/sn89075140/issues/1920.json
Status: 403
Response: <!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal


In [7]:
import zipfile
import io

# Step 1: Get the full batch list from Chronicling America
BATCH_LIST_URL = "https://chroniclingamerica.loc.gov/batches.json"

print("Fetching batch list...")
r = requests.get(BATCH_LIST_URL, timeout=30)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    data = r.json()
    batches = data.get('batches', [])
    print(f"Total batches: {len(batches)}")
    if batches:
        print(f"\nSample batch structure:")
        print(json.dumps(batches[0], indent=2))
else:
    print(f"Response: {r.text[:300]}")

Fetching batch list...
Status: 404
Response: <!DOCTYPE html>


<html lang="en" class="no-js" prefix="lc: http://loc.gov/#">
<head>


<title>Page Not Found -- 404 -- Library of Congress</title>



    




<link href='https://www.loc.gov/static/stylesheets/base.css'
    type='text/css'
    rel='stylesheet' />
<link href="https://www.loc.gov/sta


In [9]:
# Inspect resource structure of first result
data = fetch_pages_for_state_year('ohio', 1921, 3)
results = data.get('results', [])

item = results[0]
print("--- Full item ---")
print(json.dumps(item, indent=2)[:3000])

--- Full item ---
{
  "access_restricted": false,
  "aka": [
    "http://www.loc.gov/resource/sn87076601/1921-07-27/ed-1/?sp=1"
  ],
  "batch": [
    "ohi_goyle_ver01"
  ],
  "campaigns": [],
  "composite_location": [
    "0/united states/",
    "1/united states/ohio/",
    "2/united states/ohio/crawford/",
    "3/united states/ohio/crawford/bucyrus/"
  ],
  "contributor": [
    "ohio history connection, columbus, oh"
  ],
  "date": "1921-07-27",
  "dates": [
    "1921-07-27"
  ],
  "description": [
    "t D Cimo WEATHERThunderstors this afternees er tomight somewhat cooler fa the north portion partly cloudy Thursdays somewhat lower temperature I I 1 T Bt ITr caaweO y6H5s we The Bu\u00f6ukus InmingFT\u00e4leMM 4 ALL TWB NEWS AS IT HAPPENS u 3 7 t 3 Volume LXVin No 85 Whole No 10500 BUCYRUS OHIO WEDNESDAY JULY 27 1921 Thirtyfourth Year Price Three Cents REAAIIAp opreet pIAur ORure ropvnu HUULLAI OF LIT IduNI oAVeo run YUU _ ___r _ ii WARRANTS ARE ISSUED w Sheriff Mester of Springfield S